# Notebook 3: Lineares Regressionsmodell

In diesem Notebook trainieren wir ein lineares Regressionsmodell zur Vorhersage des Stundenlohns (`wage`).  
Dafür laden wir die zuvor aufbereiteten Trainings-, Validierungs- und Testdaten, trainieren das Modell und bewerten dessen Leistung.

## 1. Bibliotheken laden
Wir importieren die notwendigen Python-Bibliotheken für das Modelltraining und die spätere Evaluation.

In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## 2. Trainings-, Validierungs- und Testdaten laden
Hier laden wir die bereits gesplitteten Datensätze, die im vorherigen Notebook erstellt wurden.
Die Daten bestehen aus:
- **X** → Eingangsvariablen (Features)
- **y** → Zielvariable (Stundenlohn)

In [7]:
# Trainingsdaten
X_train = pd.read_csv('../data/Cleaned/Train70/X_train_wage_data.csv')
y_train = pd.read_csv('../data/Cleaned/Train70/y_train_wage_data.csv')

# Validationsdaten
X_val = pd.read_csv('../data/Cleaned/Validate15/X_val_wage_data.csv')
y_val = pd.read_csv('../data/Cleaned/Validate15/y_val_wage_data.csv')

# Testdaten
X_test = pd.read_csv('../data/Cleaned/Test15/X_test_wage_data.csv')
y_test = pd.read_csv('../data/Cleaned/Test15/y_test_wage_data.csv')

## 3. Überprüfung der Datenformen (Shapes)
Wir kontrollieren, ob die Datensätze korrekt geladen wurden und ob die Dimensionen stimmen.
- `X` enthält sechs Feature-Spalten
- `y` enthält die Zielgröße (wage)

In [8]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(368, 6) (368, 1)
(79, 6) (79, 1)
(79, 6) (79, 1)


## 4. Lineares Regressionsmodell trainieren
Wir initialisieren ein `LinearRegression`-Modell und trainieren es auf den Trainingsdaten (`X_train`, `y_train`).

In [11]:
from sklearn.linear_model import LinearRegression

# Modell initialisieren
lin_reg = LinearRegression()

# Modell mit Trainingsdaten trainieren
lin_reg.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## 5. Modellbewertung (Training vs. Validation)
Wir berechnen den Fehler (MSE/RMSE) und das Bestimmtheitsmaß R² auf Trainings- und Validierungsdaten.
So sehen wir, wie gut das Modell passt und ob es evtl. über- oder unterfitten könnte.

In [12]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Vorhersagen
y_train_pred = lin_reg.predict(X_train)
y_val_pred   = lin_reg.predict(X_val)

# Fehlermaße
mse_train = mean_squared_error(y_train, y_train_pred)
mse_val   = mean_squared_error(y_val,   y_val_pred)

rmse_train = np.sqrt(mse_train)
rmse_val   = np.sqrt(mse_val)

r2_train = r2_score(y_train, y_train_pred)
r2_val   = r2_score(y_val,   y_val_pred)

print("Training:")
print(f"  RMSE: {rmse_train:.3f}")
print(f"  R²  : {r2_train:.3f}")

print("\nValidation:")
print(f"  RMSE: {rmse_val:.3f}")
print(f"  R²  : {r2_val:.3f}")

Training:
  RMSE: 2.912
  R²  : 0.366

Validation:
  RMSE: 2.809
  R²  : 0.390


## 6. Modellkoeffizienten
Die Koeffizienten zeigen, wie sich der Stundenlohn im Durchschnitt verändert,
wenn ein Feature um eine Einheit steigt (bei konstanten anderen Variablen).

In [14]:
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Koeffizient": lin_reg.coef_[0]
})

print("Intercept (Basislohn):", lin_reg.intercept_)
coef_df

Intercept (Basislohn): [-2.1677808]


,Feature,Koeffizient
0,educ,0.583396
1,exper,0.026222
2,tenure,0.152667
3,nonwhite,-0.131476
4,female,-1.621042
5,married,0.350028


## 6. Modellkoeffizienten

Die Koeffizienten der linearen Regression zeigen, wie sich der vorhergesagte Stundenlohn verändert,
wenn ein Feature um eine Einheit steigt – bei konstant gehaltenen anderen Variablen.

Der **Intercept (Achsenabschnitt)** stellt den theoretischen Basislohn dar, falls alle Features den Wert 0 hätten.
Dieser Wert ist mathematisch notwendig, aber wirtschaftlich nicht real interpretierbar.

### Interpretation der Koeffizienten

- **educ (0.58)**  
  Ein zusätzliches Ausbildungsjahr erhöht den Stundenlohn im Durchschnitt um etwa **0.58 Einheiten**.

- **exper (0.03)**  
  Jedes zusätzliche Jahr Berufserfahrung steigert den Lohn leicht um ungefähr **0.03 Einheiten**.

- **tenure (0.15)**  
  Ein weiteres Jahr im gleichen Unternehmen erhöht den Lohn im Durchschnitt um **0.15 Einheiten**.

- **nonwhite (-0.13)**  
  Personen, die als „nonwhite“ klassifiziert sind, verdienen laut Modell durchschnittlich  
  **0.13 Einheiten weniger** als „white“-Personen.

- **female (-1.62)**  
  Frauen verdienen im Durchschnitt **1.62 Einheiten weniger** als Männer – selbst bei gleichen sonstigen Merkmalen.

- **married (0.35)**  
  Verheiratete Personen verdienen im Schnitt **0.35 Einheiten mehr**.

### Zusammenfassung

Die Koeffizienten zeigen, welche Variablen den größten Einfluss auf den Stundenlohn haben.
Negative Werte (z. B. für *female* oder *nonwhite*) weisen auf potenzielle Verzerrungen oder Ungleichheiten im Datensatz hin
und sollten nicht kausal interpretiert werden.

## 7. Finale Evaluation auf dem Testdatensatz
Zum Schluss bewerten wir das Modell auf den zuvor komplett ungenutzten Testdaten.

In [15]:
y_test_pred = lin_reg.predict(X_test)

mse_test  = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, y_test_pred)

print("Testdaten:")
print(f"  RMSE: {rmse_test:.3f}")
print(f"  R²  : {r2_test:.3f}")

Testdaten:
  RMSE: 3.183
  R²  : 0.324
